# 🎬 Remote AI Video Generation Pipeline on Kaggle (Free 2x Tesla T4 GPUs)
### 10 Academy AI Heritage Short Film Project (*Ato Berhanu: The Frequency of Memory*)

This notebook runs the remote compute stage of the pipeline:
1. Ingests the validated  and photorealistic FLUX keyframes.
2. Solves the 12GB 3D VAE Convolution Out-of-Memory (OOM) error using spatial VAE tiling & CPU offloading.
3. Generates 24fps motion video clips for all 9 shots.
4. Extends and stitches scenes to match exact narration timings with a 2.5s AI disclosure end card.

**Kaggle Settings Required**:
- **Accelerator**: 
- **Internet**: 

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install modern Video Diffusion dependencies
!pip install -q "diffusers>=0.31.0" "transformers>=4.45.0" "accelerate>=0.34.0" sentencepiece protobuf torchvision ftfy

In [ ]:
# 3. Locate & extract uploaded keyframes and storyboard regardless of Kaggle subfolder naming
import os, shutil, glob

print('Scanning /kaggle/input/ for uploaded dataset files...')
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        full_p = os.path.join(root, f)
        if f.endswith('.zip'):
            print(f'Unzipping: {full_p}')
            !unzip -o -q "{full_p}" -d .
        elif f.startswith('keyframe_shot_') or f == 'storyboard.json':
            shutil.copy(full_p, '.')

!ls -lh keyframe_shot_*.png storyboard.json

import json
with open('storyboard.json', 'r', encoding='utf-8') as f:
    sb = json.load(f)
print(f"✓ Successfully loaded storyboard: '{sb['title']}' ({len(sb['shots'])} shots)")

In [ ]:
# 4. Initialize Video Diffusion Pipeline with VAE Tiling (OOM Prevention)
import gc, torch
from PIL import Image
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

# Flush memory
gc.collect()
torch.cuda.empty_cache()

print('Loading LTX-Video Pipeline with bfloat16 and CPU offload...')
pipe = LTXImageToVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()

# CRITICAL FIX: Enable VAE tiling & slicing to prevent conv3d OOM (allocating 11.55 GiB)
if hasattr(pipe, 'vae'):
    pipe.vae.enable_tiling()
    pipe.vae.enable_slicing()

print('✓ Video pipeline loaded and memory-optimized for Kaggle 2x T4!')

In [ ]:
# 5. Generate 24fps motion video clips for all 9 shots
os.makedirs('generated_shots', exist_ok=True)

for i, shot in enumerate(sb['shots']):
    shot_id = shot.get('id') or shot.get('shot_id') or f"shot_{shot.get('shot_number', i+1):02d}"
    out_mp4 = f'generated_shots/{shot_id}.mp4'
    
    if os.path.exists(out_mp4):
        print(f'[{i+1}/9] {shot_id} already exists, skipping.')
        continue
        
    print(f'\n--- Generating AI Motion: Shot {i+1}/9 ({shot_id}) ---')
    candidates = [f for f in os.listdir('.') if f.startswith(f"keyframe_shot_{i+1:02d}") and f.endswith('.png')]
    if not candidates:
        print(f'Warning: Keyframe for shot {i+1} not found.')
        continue
    
    init_image = Image.open(candidates[0]).convert('RGB').resize((704, 480))
    prompt = shot.get('motion_prompt', '') + ', cinematic motion, realistic 24fps documentary short'
    
    torch.cuda.empty_cache()
    video = pipe(
        image=init_image,
        prompt=prompt,
        negative_prompt='blurry, distorted, morphing, low quality, static, artifacts',
        width=704,
        height=480,
        num_frames=25,
        num_inference_steps=25,
        guidance_scale=3.0
    ).frames[0]
    
    export_to_video(video, out_mp4, fps=24)
    print(f'✓ Saved: {out_mp4}')

print('\n🎉 All 9 AI motion shots generated successfully!')

In [ ]:
# 6. Package raw clips into zip bundle for pipeline archival
!zip -r shots_ai_motion.zip generated_shots/
!ls -lh shots_ai_motion.zip

In [ ]:
# 7. Extend shots to exact storyboard duration & append 2.5s AI End Card
import subprocess

extended_clips = []
for i, shot in enumerate(sb['shots']):
    shot_id = shot.get('id') or f"shot_{shot.get('shot_number', i+1):02d}"
    dur = shot.get('duration_seconds', 8.0)
    src_clip = f'generated_shots/{shot_id}.mp4'
    ext_clip = f'generated_shots/ext_{shot_id}.mp4'
    
    cmd = [
        'ffmpeg', '-y', '-stream_loop', '-1', '-i', src_clip,
        '-t', str(dur),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', '24',
        ext_clip
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    extended_clips.append(ext_clip)

with open('concat_list.txt', 'w') as f:
    for c in extended_clips:
        f.write(f"file '{c}'\n")

!ffmpeg -y -f concat -safe 0 -i concat_list.txt -c copy stitched_raw.mp4

# 2.5s AI Disclosure End Card
!ffmpeg -y -f lavfi -i color=c=black:s=704x480:d=2.5 \
  -vf "drawtext=text='Made with Artificial Intelligence':fontcolor=white:fontsize=24:x=(w-text_w)/2:y=(h-text_h)/2-15,drawtext=text='10 Academy Heritage Short Film Project':fontcolor=gray:fontsize=16:x=(w-text_w)/2:y=(h-text_h)/2+15" \
  -c:v libx264 -pix_fmt yuv420p -r 24 end_card.mp4

with open('final_concat.txt', 'w') as f:
    f.write("file 'stitched_raw.mp4'\nfile 'end_card.mp4'\n")

!ffmpeg -y -f concat -safe 0 -i final_concat.txt -c copy full_video_track.mp4
print('✓ full_video_track.mp4 is ready for download!')